In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("loan.csv")

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.dtypes

# EDA

In [ ]:
df.shape

In [ ]:
df.duplicated().sum()

In [ ]:
df.columns = df.columns.str.strip() #this is done because all the columns are initially having spaces before them

In [ ]:
sns.countplot(x="loan_status", data=df, palette="Set2")
plt.title("Loan Status Distribution")
plt.show()


In [ ]:
num_cols = ["income_annum","loan_amount","loan_term","cibil_score",
            "residential_assets_value","commercial_assets_value",
            "luxury_assets_value","bank_asset_value"]

In [ ]:
df[num_cols].hist(bins=30, figsize=(15,10))
plt.suptitle("Distribution of Numeric Features")
plt.show()

In [ ]:
# Boxplots vs target
for col in ["income_annum","loan_amount","cibil_score"]:
    plt.figure(figsize=(6,4))
    sns.boxplot(x="loan_status", y=col, data=df, palette="Set3")
    plt.title(f"{col} vs Loan Status")
    plt.show()

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df[num_cols].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap of Numeric Features")
plt.show()

In [ ]:
cat_cols = ["education","self_employed"]

for col in cat_cols:
    plt.figure(figsize=(6,4))
    sns.countplot(x=col, data=df, palette="Set1")
    plt.title(f"Distribution of {col}")
    plt.show()

    plt.figure(figsize=(6,4))
    sns.countplot(x=col, hue="loan_status", data=df, palette="Set2")
    plt.title(f"{col} vs Loan Status")
    plt.show()

In [ ]:
# Income vs Loan Amount
plt.figure(figsize=(6,4))
sns.scatterplot(x="income_annum", y="loan_amount", hue="loan_status", data=df)
plt.title("Income vs Loan Amount")
plt.show()

# Cibil Score vs Loan Status
sns.violinplot(x="loan_status", y="cibil_score", data=df, palette="Set3")
plt.title("Cibil Score vs Loan Status")
plt.show()

# Total Assets vs Loan Status
df["total_assets"] = df["residential_assets_value"] + df["commercial_assets_value"] + df["luxury_assets_value"] + df["bank_asset_value"]

sns.boxplot(x="loan_status", y="total_assets", data=df, palette="Set2")
plt.title("Total Assets vs Loan Status")
plt.show()


# Feature Engineering

***1 . Encoding***

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode categorical columns
cat_cols = ["education", "self_employed", "loan_status"]
le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print(df[cat_cols].head())


***2 .Scaling***

In [ ]:
from sklearn.preprocessing import StandardScaler

num_cols = ["income_annum","loan_amount","loan_term","cibil_score",
            "residential_assets_value","commercial_assets_value",
            "luxury_assets_value","bank_asset_value"]

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

print(df[num_cols].head())


In [ ]:
import joblib
joblib.dump(X.columns.tolist(), "feature_order.pkl")

***3. Derived Features***

In [ ]:
# Debt-to-income ratio
df["debt_income_ratio"] = df["loan_amount"] / (df["income_annum"] + 1)  # avoid division by zero

# Total assets
df["total_assets"] = (df["residential_assets_value"] + df["commercial_assets_value"] +
                      df["luxury_assets_value"] + df["bank_asset_value"])

print(df[["debt_income_ratio","total_assets"]].head())


In [ ]:
X = df.drop(columns=["loan_id","loan_status"])  # predictors
y = df["loan_status"]                           # target


***4. Train Test Split***

In [ ]:
from sklearn.model_selection import train_test_split

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


***5. Baseline Model (Logistic Regression)***

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
log_reg = LogisticRegression(max_iter=1000)

In [ ]:
log_reg.fit(X_train, y_train)

In [ ]:
# Predictions
y_pred = log_reg.predict(X_test)
y_prob = log_reg.predict_proba(X_test)[:,1]

***6. Model Evaluation***

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Rejected","Approved"], yticklabels=["Rejected","Approved"])
plt.title("Confusion Matrix")
plt.show()

***7 . Feature Importance(Coefficients)***

In [ ]:
import numpy as np

coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": log_reg.coef_[0]
}).sort_values(by="Coefficient", ascending=False)

print(coefficients)


# Training with Decision Tree Algorithm

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
# Initialize model
dt_model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=None,
    random_state=42
)

In [ ]:
dt_model.fit(X_train, y_train)

In [ ]:
# Predictions
y_pred_dt = dt_model.predict(X_test)
y_prob_dt = dt_model.predict_proba(X_test)[:,1]

***Evaluation***

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Precision:", precision_score(y_test, y_pred_dt))
print("Recall:", recall_score(y_test, y_pred_dt))
print("F1 Score:", f1_score(y_test, y_pred_dt))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_dt))

# Confusion Matrix
cm_dt = confusion_matrix(y_test, y_pred_dt)
sns.heatmap(cm_dt, annot=True, fmt="d", cmap="Greens", xticklabels=["Rejected","Approved"], yticklabels=["Rejected","Approved"])
plt.title("Decision Tree Confusion Matrix")
plt.show()


***Feature Importance***

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": dt_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(feature_importance)


***visualizing the tree***

In [ ]:
from sklearn import tree
plt.figure(figsize=(20,10))
tree.plot_tree(dt_model, feature_names=X.columns, class_names=["Rejected","Approved"], filled=True, fontsize=8)
plt.show()

# Training with Random Forest Algorithm

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    criterion="gini",
    max_depth=None,
    random_state=42
)

In [ ]:
# Train
rf_model.fit(X_train, y_train)

In [ ]:
# Predictions
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:,1]

***Model Evaluation***

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

# Confusion Matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Oranges", xticklabels=["Rejected","Approved"], yticklabels=["Rejected","Approved"])
plt.title("Random Forest Confusion Matrix")
plt.show()

***Feature Importance***

In [ ]:
feature_importance_rf = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(feature_importance_rf)

# Training with XGboost (Default Parameters)

In [ ]:
from xgboost import XGBClassifier

In [ ]:
xgb_model = XGBClassifier(
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)

In [ ]:
xgb_model.fit(X_train, y_train)

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:,1]

***Model Evaluation***

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall:", recall_score(y_test, y_pred_xgb))
print("F1 Score:", f1_score(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))

In [ ]:
# Confusion Matrix
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm_xgb, annot=True, fmt="d", cmap="Purples", xticklabels=["Rejected","Approved"], yticklabels=["Rejected","Approved"])
plt.title("XGBoost Confusion Matrix")
plt.show()

***Feature Importance***

In [ ]:
feature_importance_xgb = pd.DataFrame({
    "Feature": X.columns,
    "Importance": xgb_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(feature_importance_xgb)

***hyperparameter tuning for XGBoost***

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

In [ ]:
#base model
xgb_base = XGBClassifier(
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)

In [ ]:
# Parameter grid for GridSearchCV
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

***GridSearchCV***

In [ ]:
grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    verbose=2,
    n_jobs=-1
)

In [ ]:
grid_search.fit(X_train, y_train)

In [ ]:
print("Best Parameters (GridSearchCV):", grid_search.best_params_)
print("Best F1 Score:", grid_search.best_score_)

***RandomizedSearchCV***

In [ ]:
param_dist = {
    "n_estimators": [100, 200, 300, 400, 500],
    "max_depth": [3, 4, 5, 6, 7, 8],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0, 1, 5]   # minimum loss reduction for splits
}

In [ ]:
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=20,           # number of random combinations
    scoring="f1",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

In [ ]:
random_search.fit(X_train, y_train)

In [ ]:
print("Best Parameters (RandomizedSearchCV):", random_search.best_params_)
print("Best F1 Score:", random_search.best_score_)

# Train Final Model with Best Params

In [ ]:
best_params = grid_search.best_params_

In [ ]:

xgb_final = XGBClassifier(
    **best_params,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)

In [ ]:
xgb_final.fit(X_train, y_train)

In [ ]:
# Evaluate again
y_pred_final = xgb_final.predict(X_test)
y_prob_final = xgb_final.predict_proba(X_test)[:,1]

In [ ]:

print("Final Accuracy:", accuracy_score(y_test, y_pred_final))
print("Final Precision:", precision_score(y_test, y_pred_final))
print("Final Recall:", recall_score(y_test, y_pred_final))
print("Final F1 Score:", f1_score(y_test, y_pred_final))
print("Final ROC-AUC:", roc_auc_score(y_test, y_prob_final))

# Save the model

In [ ]:
import joblib

In [ ]:
# Save model
joblib.dump(xgb_final, "loan_approval_model.pkl")